# Track B - official retrain, epochs 17-24

Resumes **Oriented R-CNN (ConvNeXt-S + PAFPN, Gaussian KLD, ClassBalancedDataset)** at epoch 16 and
trains epochs 17-24 on the **official DOTA v2.0 trainval split** (1,830 images, 18 classes, difficulty
flags preserved), then re-evaluates on the official 7,122-patch validation tiling.

Colab now runs **Python 3.13**, where torch 2.0.1 / mmcv 2.0.1 have no wheels. Cell 2 therefore
provisions the project's own **Python 3.11 venv** (scripts/setup_colab.sh, uv + pinned wheels).
Every call that needs torch/mmcv goes through that venv; the notebook kernel only orchestrates.

The recipe is not modified. Only these change, all forced by the environment:

| Item | Original | Here | Why |
|---|---|---|---|
| GPUs | 1 x A100, then 2 x T4 | 1 x T4 | available hardware |
| per-step batch | 8 (A100) / 4 per GPU (T4) | 4 x 2 accumulation | effective batch stays 8 |
| work dir | .../track_b_convnext_s_pafpn_kld_2x | .../trackB_official_ep17-24 | the old run is never touched |
| framework | torch 2.0.1, mmcv 2.0.1, mmdet 3.1.0 | identical | pinned venv |

Validation cadence is the original val_interval=2, so epochs 18, 20, 22 and 24 are validated.

**Run order:** 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7. After a disconnect, re-run 1, 2 (it is a no-op if the
venv exists), 3, 5 - the work dir and checkpoints persist on Drive.

In [ ]:
#@title 1 - Clone the repo, show the GPU and the system Python { display-mode: "form" }
import subprocess, sys
from pathlib import Path

REPO = Path('/content/ADA')
COMMIT = 'a2ecd69d3d487c3f639c3cec329e0a737454370e'

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
print('system python', sys.version.split()[0], '->', sys.executable)
df = subprocess.run(['df', '-BG', '/content'], capture_output=True, text=True).stdout.splitlines()[-1]
print('disk', df)

if not (REPO / 'scripts').is_dir():
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/Lion4467cat/dotav2-obb-benchmark.git',
                    str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', COMMIT], check=True)
print('repo at', subprocess.run(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'],
                                capture_output=True, text=True).stdout.strip())

assert subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0, 'no GPU on this runtime'

In [ ]:
#@title 2 - Provision the Python 3.11 venv with the pinned OpenMMLab stack { display-mode: "form" }
import subprocess, time
from pathlib import Path

REPO = Path('/content/ADA')
VENV = Path('/content/mmrotate_venv')

if (VENV / 'bin' / 'python').is_file():
    print('venv already present at', VENV)
else:
    t0 = time.time()
    p = subprocess.Popen(['bash', str(REPO / 'scripts' / 'setup_colab.sh'), str(REPO), str(VENV)],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    print()
    print('setup exit', p.returncode, 'after', round((time.time() - t0) / 60, 1), 'min')
    assert p.returncode == 0, 'setup_colab.sh failed'
print('venv python', (VENV / 'bin' / 'python').resolve())

In [ ]:
#@title 3 - Verify the venv sees the GPU, mount Drive, stage epoch_16.pth { display-mode: "form" }
import subprocess, os
from pathlib import Path

REPO = Path('/content/ADA')
VENV = Path('/content/mmrotate_venv')
PY = VENV / 'bin' / 'python'

PROBE = '''
import torch, mmcv, mmengine, mmdet, os, mmrotate
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.version.cuda)
print('device', torch.cuda.get_device_name(0),
      '| vram GiB', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print('mmcv', mmcv.__version__, '| mmengine', mmengine.__version__, '| mmdet', mmdet.__version__)
print('mmrotate', os.path.dirname(mmrotate.__file__))
'''
r = subprocess.run([str(PY), '-c', PROBE], capture_output=True, text=True)
print(r.stdout)
assert r.returncode == 0, r.stderr[-2000:]

from google.colab import drive
mounted = False
for attempt in range(3):
    try:
        drive.mount('/content/drive', force_remount=(attempt > 0))
        mounted = True
        break
    except Exception as exc:
        print('drive mount attempt', attempt + 1, 'failed:', type(exc).__name__, exc)
print('drive mounted:', mounted)

MYDRIVE = Path('/content/drive/MyDrive')
if mounted and MYDRIVE.is_dir():
    OUT = MYDRIVE / 'ada_trackB_official'
else:
    OUT = Path('/content/ada_trackB_official')
    print('WARNING: Drive unavailable - writing to', OUT, '(lost if the session dies)')
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'logs').mkdir(exist_ok=True)

CKPT16 = Path('/content/epoch_16.pth')
if not CKPT16.is_file() or CKPT16.stat().st_size < 700e6:
    source = None
    if mounted:
        for cand in (MYDRIVE / 'dotav2_benchmark').rglob('epoch_16.pth'):
            if cand.is_file() and cand.stat().st_size > 700e6:
                source = cand
                break
    if source is not None:
        print('copying checkpoint from', source)
        subprocess.run(['cp', str(source), str(CKPT16)], check=True)
    else:
        print('downloading epoch_16.pth by Drive file id (gdown)')
        subprocess.run([str(VENV / 'bin' / 'gdown'),
                        '12I5f7mNGUGJpMCIVKBLh1IekMhDkrpPs', '-O', str(CKPT16)], check=True)
print('checkpoint', CKPT16, round(CKPT16.stat().st_size / 1e6), 'MB')
print('output root', OUT)

In [ ]:
#@title 4 - Official DOTA v2.0 -> 1024/200 tiling, then PROVE it is the right dataset { display-mode: "form" }
import subprocess, time
from collections import Counter
from pathlib import Path

SPLIT = Path('/content/data/split_ss_dotav2')
RAW = Path('/content/data/dotav2_raw')
CACHE = Path('/content/data/raw_archives')
PY = Path('/content/mmrotate_venv/bin/python')

def sh(cmd, cwd=None):
    print('+', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.run([str(c) for c in cmd], cwd=str(cwd) if cwd else None)
    assert p.returncode == 0, 'command failed: ' + ' '.join(str(c) for c in cmd)

t0 = time.time()
have_patches = (SPLIT / 'train' / 'annfiles').is_dir() and any((SPLIT / 'train' / 'images').glob('*.png'))
if not have_patches:
    sh([PY, REPO / 'scripts' / 'download_dotav2.py', '--source', 'huggingface',
        '--cache-dir', CACHE, '--out-dir', RAW], cwd=REPO)
    sh([PY, REPO / 'scripts' / 'prepare_dotav2_patches.py', '--raw-dir', RAW, '--out-dir', SPLIT,
        '--mmrotate-dir', REPO / 'mmrotate', '--patch-size', '1024', '--overlap', '200',
        '--nproc', '4'], cwd=REPO)
print('data ready in', round((time.time() - t0) / 60, 1), 'min')

def scan(ann_dir):
    classes, flags, inst = Counter(), Counter(), 0
    files = list(Path(ann_dir).glob('*.txt'))
    for f in files:
        for line in f.read_text(errors='ignore').splitlines():
            p = line.split()
            if len(p) < 10 or p[0].lower() in ('imagesource:', 'gsd:'):
                continue
            classes[p[8]] += 1
            flags[p[9]] += 1
            inst += 1
    return len(files), inst, classes, flags

V2_CLASSES = {'plane', 'ship', 'storage-tank', 'baseball-diamond', 'tennis-court',
              'basketball-court', 'ground-track-field', 'harbor', 'bridge', 'large-vehicle',
              'small-vehicle', 'helicopter', 'roundabout', 'soccer-ball-field',
              'swimming-pool', 'container-crane', 'airport', 'helipad'}

# ---- source-level provenance: the official 18-class annotations, difficulty flags intact ----
def ann_root(kind):
    # the tiler consumes RAW/<split>/labelTxt, so that is what must be proven official
    direct = RAW / kind / 'labelTxt'
    if direct.is_dir():
        return direct
    hits = [p for p in RAW.rglob('version2.0') if p.is_dir() and kind in str(p)]
    return hits[0] if hits else None

src_train, src_val = ann_root('train'), ann_root('val')
print('labels that feed the tiler:', src_train, '|', src_val)
tr_f, tr_i, tr_c, tr_fg = scan(src_train)
va_f, va_i, va_c, va_fg = scan(src_val)
print('SOURCE train files', tr_f, 'instances', tr_i, 'classes', len(tr_c))
print('SOURCE val   files', va_f, 'instances', va_i, 'classes', len(va_c))
print('SOURCE train difficulty flags', dict(tr_fg))

# ---- patch-level: what the network will actually see ----
p_tr_f, p_tr_i, p_tr_c, _ = scan(SPLIT / 'train' / 'annfiles')
p_va_f, p_va_i, p_va_c, p_va_fg = scan(SPLIT / 'val' / 'annfiles')
n_tr_img = len(list((SPLIT / 'train' / 'images').glob('*.png')))
n_va_img = len(list((SPLIT / 'val' / 'images').glob('*.png')))
print()
print('PATCH train', n_tr_img, 'images /', p_tr_f, 'annfiles /', p_tr_i, 'instances')
print('PATCH val  ', n_va_img, 'images /', p_va_f, 'annfiles /', p_va_i, 'instances')
print('PATCH val non-difficult', p_va_fg.get('0', 0), '| difficult', p_va_fg.get('1', 0))
print('PATCH classes', len(p_tr_c), sorted(p_tr_c))

checks = []
def check(name, ok, detail=''):
    checks.append((name, bool(ok), detail))
    print(('  PASS  ' if ok else '  FAIL  ') + name + ('   ' + str(detail) if detail else ''))

print()
check('source train annfiles == 1830', tr_f == 1830, tr_f)
check('source train instances == 268627', tr_i == 268627, tr_i)
check('source val annfiles == 593', va_f == 593, va_f)
check('18 classes at source level', len(tr_c) == 18, len(tr_c))
check('class set is DOTA v2.0 (airport/helipad/container-crane present)',
      set(tr_c) == V2_CLASSES, sorted(V2_CLASSES - set(tr_c)) or 'exact match')
check('difficulty flags present at source level',
      tr_fg.get('0', 0) > 0 and tr_fg.get('1', 0) > 0,
      'flag0=' + str(tr_fg.get('0', 0)) + ' flag1=' + str(tr_fg.get('1', 0)))
check('val patches == 7122', n_va_img == 7122, n_va_img)
check('val non-difficult instances == 69542', p_va_fg.get('0', 0) == 69542, p_va_fg.get('0', 0))
check('val difficult instances == 70561', p_va_fg.get('1', 0) == 70561, p_va_fg.get('1', 0))
check('train patches in [12800, 13500]', 12800 <= n_tr_img <= 13500, n_tr_img)
check('18 classes at patch level', len(p_tr_c) == 18, len(p_tr_c))

tr_names = {p.stem for p in (SPLIT / 'train' / 'images').glob('*.png')}
va_names = {p.stem for p in (SPLIT / 'val' / 'images').glob('*.png')}
overlap = tr_names & va_names
check('no train/val patch overlap', len(overlap) == 0, len(overlap))

failed = [n for n, ok, _ in checks if not ok]
assert not failed, 'DATASET VERIFICATION FAILED: ' + ', '.join(failed)
print()
print('dataset verification: all', len(checks), 'checks passed')

In [ ]:
#@title 5 - TRAIN epochs 17-24 (re-run after any disconnect) { display-mode: "form" }
import subprocess, time, os
from pathlib import Path

PY = Path('/content/mmrotate_venv/bin/python')
WORK = Path('/content/work_dirs/trackB_official_ep17-24')
WORK.mkdir(parents=True, exist_ok=True)
DRIVE_WORK = OUT / 'work_dirs' / 'trackB_official_ep17-24'
DRIVE_WORK.mkdir(parents=True, exist_ok=True)

# ---- the exact epoch-16 config, straight out of the checkpoint ----
CFG16 = WORK / 'ep16_cfg.py'
if not CFG16.is_file():
    code = ("import torch, pathlib;"
            "ck = torch.load('/content/epoch_16.pth', map_location='cpu');"
            "pathlib.Path('" + str(CFG16) + "').write_text(ck['meta']['cfg']);"
            "print('config extracted, chars', len(ck['meta']['cfg']))")
    subprocess.run([str(PY), '-c', code], check=True)

TRAIN_CFG = WORK / 'train_cfg.py'
base = CFG16.read_text()
extra = '''
# ---- official-retrain overrides; the recipe above is untouched ----
work_dir = '%s'
resume = True
load_from = '/content/epoch_16.pth'
train_dataloader['batch_size'] = 4
optim_wrapper['accumulative_counts'] = 2
train_cfg['val_interval'] = 2
default_hooks['checkpoint'].update(dict(interval=1, max_keep_ckpts=3, save_best='dota/mAP', out_dir='%s'))
custom_hooks = list(globals().get('custom_hooks', [])) + [
    dict(type='CheckpointHook', interval=1, out_dir='%s', max_keep_ckpts=2),
]
''' % (WORK, WORK, DRIVE_WORK)
TRAIN_CFG.write_text(base + extra)
print('training config ->', TRAIN_CFG)

env = dict(os.environ)
env['MPLBACKEND'] = 'Agg'
env['PYTHONPATH'] = str(REPO) + ':' + str(REPO / 'mmrotate') + ':' + env.get('PYTHONPATH', '')
env['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

# MMEngine's --resume reads work_dir/last_checkpoint; point it at epoch 16 on a fresh work dir
if not any(WORK.glob('epoch_*.pth')) and not any(WORK.glob('iter_*.pth')):
    (WORK / 'last_checkpoint').write_text('/content/epoch_16.pth')
    print('seeded last_checkpoint ->', (WORK / 'last_checkpoint').read_text())

cmd = [str(PY), str(REPO / 'mmrotate' / 'tools' / 'train.py'), str(TRAIN_CFG),
       '--work-dir', str(WORK), '--resume']
print('+', ' '.join(cmd))
print('per-step batch 4 x 2 accumulation = effective batch 8; expected LR at epoch 17 = 2.0366e-05')

LOG = WORK / 'train_stdout.log'
t0 = time.time()
with open(LOG, 'a') as log:
    p = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    p.wait()
elapsed = (time.time() - t0) / 3600
print()
print('train.py exit', p.returncode, 'after', round(elapsed, 2), 'h')
print('checkpoints:', sorted(x.name for x in WORK.glob('*.pth')))
assert p.returncode == 0, 'training failed - see the log above'

In [ ]:
#@title 6 - Final evaluation on the official 7,122-patch val tiling { display-mode: "form" }
import subprocess, json, os, time
from pathlib import Path

PY = Path('/content/mmrotate_venv/bin/python')
WORK = Path('/content/work_dirs/trackB_official_ep17-24')
SPLIT = Path('/content/data/split_ss_dotav2')

cands = sorted(WORK.glob('epoch_*.pth'), key=lambda p: int(p.stem.split('_')[-1]))
assert cands, 'no epoch checkpoint in ' + str(WORK)
CKPT = cands[-1]
print('evaluating', CKPT, round(CKPT.stat().st_size / 1e6), 'MB')

env = dict(os.environ)
env['MPLBACKEND'] = 'Agg'
env['PYTHONPATH'] = str(REPO) + ':' + str(REPO / 'mmrotate') + ':' + env.get('PYTHONPATH', '')
env['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

cmd = [str(PY), str(REPO / 'scripts' / 'run_track_eval.py'),
       '--config', str(WORK / 'train_cfg.py'),
       '--work-dir', str(WORK),
       '--checkpoint', str(CKPT),
       '--ann-dir', str(SPLIT / 'val' / 'annfiles'),
       '--batch-size', '4']
print('+', ' '.join(cmd))
t0 = time.time()
p = subprocess.run(cmd, cwd=str(REPO), env=env, capture_output=True, text=True)
out = (p.stdout or '') + (p.stderr or '')
Path(OUT / 'logs' / 'eval_stdout.log').write_text(out)
print(out[-4000:])
assert p.returncode == 0, 'evaluation failed'

EXT = WORK / 'eval_test' / 'extended_metrics.json'
metrics = json.loads(EXT.read_text())
print('extended metrics:', EXT)
for k in ('num_images', 'num_gt_non_difficult', 'num_predictions', 'map50', 'ap75', 'map50_95',
          'class_agnostic_ap50', 'classification_gap_ap50'):
    print('  ', k, '=', metrics[k])
print('   operational =', metrics['operational'])
print('   difficulty_recall =', metrics['difficulty_recall'])
print('elapsed', round((time.time() - t0) / 60, 1), 'min')

In [ ]:
#@title 7 - Comparison table, results JSON, deviations { display-mode: "form" }
import json
from pathlib import Path

WORK = Path('/content/work_dirs/trackB_official_ep17-24')
metrics = json.loads((WORK / 'eval_test' / 'extended_metrics.json').read_text())

BASELINE = {'classes': ['plane', 'baseball-diamond', 'bridge', 'ground-track-field', 'small-vehicle', 'large-vehicle', 'ship', 'tennis-court', 'basketball-court', 'storage-tank', 'soccer-ball-field', 'roundabout', 'harbor', 'swimming-pool', 'helicopter', 'container-crane', 'airport', 'helipad'], 'track_a_ep10': {'map50': 0.6049, 'ap75': 0.2785, 'map50_95': 0.3146, 'class_agnostic_ap50': 0.7985, 'operational': {'fps': 11.36, 'params_m': None, 'peak_vram_gb': 2.86}, 'per_class': {'plane': {'ap50': 0.9072}, 'baseball-diamond': {'ap50': 0.6984}, 'bridge': {'ap50': 0.3901}, 'ground-track-field': {'ap50': 0.7285}, 'small-vehicle': {'ap50': 0.7653}, 'large-vehicle': {'ap50': 0.7296}, 'ship': {'ap50': 0.8237}, 'tennis-court': {'ap50': 0.9385}, 'basketball-court': {'ap50': 0.6067}, 'storage-tank': {'ap50': 0.8081}, 'soccer-ball-field': {'ap50': 0.5525}, 'roundabout': {'ap50': 0.7123}, 'harbor': {'ap50': 0.6067}, 'swimming-pool': {'ap50': 0.6367}, 'helicopter': {'ap50': 0.5346}, 'container-crane': {'ap50': 0.0}, 'airport': {'ap50': 0.4492}, 'helipad': {'ap50': 0.0}}}, 'track_b_old': {'map50': 0.6968, 'ap75': 0.4347, 'map50_95': 0.4175, 'class_agnostic_ap50': 0.8765, 'operational': {'fps': 5.7, 'params_m': 70.24, 'peak_vram_gb': 4.2}, 'per_class': {'plane': {'ap50': 0.9359}, 'baseball-diamond': {'ap50': 0.7567}, 'bridge': {'ap50': 0.5573}, 'ground-track-field': {'ap50': 0.7983}, 'small-vehicle': {'ap50': 0.822}, 'large-vehicle': {'ap50': 0.8687}, 'ship': {'ap50': 0.9111}, 'tennis-court': {'ap50': 0.957}, 'basketball-court': {'ap50': 0.7621}, 'storage-tank': {'ap50': 0.8286}, 'soccer-ball-field': {'ap50': 0.6723}, 'roundabout': {'ap50': 0.7628}, 'harbor': {'ap50': 0.8027}, 'swimming-pool': {'ap50': 0.6686}, 'helicopter': {'ap50': 0.6685}, 'container-crane': {'ap50': 0.0323}, 'airport': {'ap50': 0.4876}, 'helipad': {'ap50': 0.25}}}}

def row(name, m):
    op = m.get('operational', {})
    return (name, m['map50'], m['ap75'], m['map50_95'], m['class_agnostic_ap50'],
            op.get('fps'), op.get('params_m'), op.get('peak_vram_gb'))

A = BASELINE['track_a_ep10']
B_OLD = BASELINE['track_b_old']
lines = []
lines.append('| Metric | Track A ep10 (R50) | Track B OLD ep24 | Track B NEW ep24 | delta NEW-OLD |')
lines.append('|---|---|---|---|---|')
specs = [('mAP50', 'map50'), ('AP75', 'ap75'), ('mAP50-95', 'map50_95'),
         ('class-agnostic AP50', 'class_agnostic_ap50')]
for label, key in specs:
    a, b, n = A[key], B_OLD[key], metrics[key]
    lines.append('| %s | %.4f | %.4f | %.4f | %+.4f |' % (label, a, b, n, n - b))
lines.append('| params (M) | %.2f | %.2f | %.2f | %+.2f |' % (
    A['operational']['params_m'], B_OLD['operational']['params_m'], metrics['operational']['params_m'],
    metrics['operational']['params_m'] - B_OLD['operational']['params_m']))
table = chr(10).join(lines)

per = []
per.append('| Class | Track B OLD AP50 | Track B NEW AP50 | delta |')
per.append('|---|---|---|---|')
for cls in BASELINE['classes']:
    if cls in metrics['per_class'] and cls in B_OLD['per_class']:
        o = B_OLD['per_class'][cls]['ap50']
        n = metrics['per_class'][cls]['ap50']
        per.append('| %s | %.4f | %.4f | %+.4f |' % (cls, o, n, n - o))
per_table = chr(10).join(per)

comparison = '# Track B official retrain - comparison' + chr(10) + chr(10) + table + chr(10) + chr(10) + '## Per-class AP50' + chr(10) + chr(10) + per_table + chr(10)
(OUT / 'comparison_table.md').write_text(comparison)

# ---- arithmetic self-checks ----
per_class = metrics['per_class']
assert len(per_class) == 18, len(per_class)
mean_ap50 = sum(v['ap50'] for v in per_class.values()) / 18
mean_ap = sum(v['ap50_95'] for v in per_class.values()) / 18
print('mean per-class AP50 %.4f vs headline %.4f' % (mean_ap50, metrics['map50']))
print('mean per-class AP50-95 %.4f vs headline %.4f' % (mean_ap, metrics['map50_95']))
assert abs(mean_ap50 - metrics['map50']) < 5e-4
assert abs(mean_ap - metrics['map50_95']) < 5e-4
assert metrics['num_gt_non_difficult'] == 69542

results = {
    'model': 'oriented_rcnn_convnext_s_pafpn_gaussian_kld',
    'checkpoint': 'epoch_24.pth',
    'trained_epochs': '17-24 on official DOTA v2.0 trainval (18 classes, difficulty flags preserved)',
    'resumed_from': 'epoch_16.pth',
    'gpu': '1 x T4 16GB (Colab)',
    'per_step_batch': 4,
    'gradient_accumulation': 2,
    'effective_batch': 8,
    'metrics': {k: metrics[k] for k in ('num_images', 'num_gt_non_difficult', 'num_predictions',
                                        'map50', 'ap75', 'map50_95', 'class_agnostic_ap50',
                                        'classification_gap_ap50', 'micro_prf', 'macro_f1_opt',
                                        'scale_ap50', 'orientation_error_deg', 'difficulty_recall',
                                        'operational')},
    'per_class': per_class,
    'baseline_track_b_old': {k: B_OLD[k] for k in ('map50', 'ap75', 'map50_95', 'class_agnostic_ap50')},
    'baseline_track_a_ep10': {k: A[k] for k in ('map50', 'ap75', 'map50_95', 'class_agnostic_ap50')},
}
(OUT / 'results_trackB_official_retrain.json').write_text(json.dumps(results, indent=2))

deviations = [
    '# Deviations from the original recipe',
    '',
    'Every entry below is environment-forced. Model, loss, optimizer, LR schedule, augmentation,',
    'sampler and seed policy are unchanged.',
    '',
    '1. Hardware: 1 x T4 16GB instead of 1 x A100 then 2 x T4. Effective batch kept at 8 by',
    '   per-step batch 4 with 2-step gradient accumulation.',
    '2. Runtime: Colab ships Python 3.13, where torch 2.0.1 / mmcv 2.0.1 have no wheels. The',
    '   project venv (Python 3.11, uv) is used instead of the notebook kernel.',
    '3. work_dir is new (trackB_official_ep17-24). The original run directory is read-only input.',
    '4. Validation runs at epochs 18/20/22/24 (original val_interval=2) instead of a custom cadence.',
    '5. A second CheckpointHook mirrors each epoch to Drive so a disconnect costs at most one epoch.',
    '6. num_workers is unchanged at 4; batch/accumulation are the only dataloader-adjacent changes.',
    '',
    'Not changed: model architecture, Gaussian KLD loss, AdamW base LR 1e-4, LinearLR warmup 500',
    'iters, CosineAnnealingLR T_max 24 eta_min 1e-6, ClassBalancedDataset oversample_thr 1e-3,',
    'all augmentation pipelines, seed 1143503445.',
]
(OUT / 'deviations.md').write_text(chr(10).join(deviations))

print()
print(table)
print()
print('wrote:')
for name in ('results_trackB_official_retrain.json', 'comparison_table.md', 'deviations.md'):
    print('  ', OUT / name)
print()
print('SUMMARY_JSON', json.dumps({'map50': metrics['map50'], 'ap75': metrics['ap75'],
                                 'map50_95': metrics['map50_95'],
                                 'old_map50': B_OLD['map50'], 'old_map50_95': B_OLD['map50_95'],
                                 'fps': metrics['operational'].get('fps'),
                                 'peak_vram_gb': metrics['operational'].get('peak_vram_gb')}))